# SEC-C Mini-GAT GNN Training Notebook

**Model:** MiniGAT (2-layer GAT, 4 heads, input_dim=773, hidden_dim=256, output_dim=128)  
**Dataset:** NIST Juliet Test Suite (Java + C/C++)  
**Embeddings:** GraphCodeBERT (microsoft/graphcodebert-base, 768-dim)  
**Node Features:** 5-dim per-node structural features  
**Conformal:** Adaptive Prediction Sets (APS) with alpha=0.1  

This notebook is self-contained: all code is inline, no imports from `src/`.  
Target runtime: Kaggle GPU P100 (16GB VRAM), under 6 hours.

## Cell 1: Setup & Install

In [ ]:
# ============================================================================
# Cell 1: Setup & Install
# ============================================================================
import subprocess, sys, os, warnings

# Suppress noisy CUDA compatibility warnings (we handle this ourselves)
warnings.filterwarnings("ignore", message=".*Found GPU.*cuda capability.*")
warnings.filterwarnings("ignore", message=".*not compatible with the current PyTorch.*")
warnings.filterwarnings("ignore", message=".*Please install PyTorch.*")

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core dependencies
pip_install("torch-geometric")
pip_install("transformers")
pip_install("networkx")
pip_install("scikit-learn")

# --------------------------------------------------------------------------
# Device selection: CUDA only if GPU compute capability >= 7.0
# --------------------------------------------------------------------------
import torch

device = "cpu"

if torch.cuda.is_available():
    try:
        cap_major, cap_minor = torch.cuda.get_device_capability(0)
        gpu_name = torch.cuda.get_device_name(0)
        print(f"GPU detected: {gpu_name} (sm_{cap_major}{cap_minor})")

        if cap_major >= 7:
            device = "cuda"
            props = torch.cuda.get_device_properties(0)
            vram_gb = props.total_global_mem / (1024 ** 3)
            print(f"VRAM: {vram_gb:.1f} GB")
            print("GPU is compatible -- using CUDA acceleration.")
        else:
            print()
            print(f"  Compute capability {cap_major}.{cap_minor} is below PyTorch minimum (7.0).")
            print("  Falling back to CPU. Training will be slower but fully functional.")
            print("  TIP: On Kaggle, go to Settings > Accelerator > GPU T4 x2")
            print("       for a compatible GPU (sm_75).")
    except Exception as e:
        print(f"GPU check failed ({e}). Using CPU.")
else:
    print("No GPU detected. Training will use CPU.")

print()
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

import torch_geometric
print(f"PyG: {torch_geometric.__version__}")

print()
print("Setup complete.")


## Cell 2: Configuration

In [ ]:
# ============================================================================
# Cell 2: Configuration -- all hyperparameters in one place
# ============================================================================

CONFIG = {
    # Model architecture
    "embedding_model": "microsoft/graphcodebert-base",
    "embedding_dim": 768,
    "node_feature_dim": 5,
    "input_dim": 773,          # 768 + 5
    "hidden_dim": 256,
    "output_dim": 128,
    "num_heads_l1": 4,
    "num_heads_l2": 4,
    "dropout": 0.3,
    "num_classes": 2,

    # Data processing
    "max_nodes": 200,
    "max_tokens_per_node": 128,
    "embedding_batch_size": 64,

    # Training
    "batch_size": 32,
    "epochs": 50,
    "lr": 0.001,
    "weight_decay": 1e-4,
    "patience": 10,
    "confidence_loss_weight": 0.2,
    "grad_clip": 1.0,

    # Conformal prediction
    "alpha": 0.1,

    # Data split
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "cal_ratio": 0.15,          # calibration + test

    # Target CWEs (top 15 by sample count in Juliet)
    "target_cwes": [
        "CWE-89",   # SQL Injection
        "CWE-78",   # OS Command Injection
        "CWE-79",   # XSS
        "CWE-476",  # NULL Pointer Dereference
        "CWE-190",  # Integer Overflow
        "CWE-401",  # Memory Leak
        "CWE-122",  # Heap-Based Buffer Overflow
        "CWE-121",  # Stack-Based Buffer Overflow
        "CWE-134",  # Uncontrolled Format String
        "CWE-369",  # Divide by Zero
        "CWE-690",  # NULL Deref from Return
        "CWE-80",   # Basic XSS
        "CWE-36",   # Absolute Path Traversal
        "CWE-400",  # Uncontrolled Resource Consumption
        "CWE-197",  # Numeric Truncation Error
    ],
}

# Sink patterns for is_sink feature
SINK_PATTERNS = [
    "execute", "exec", "system", "popen", "runtime", "processbuilder",
    "sendredirect", "forward", "include", "write", "print", "println",
    "printf", "sprintf", "fprintf", "strcpy", "strcat", "memcpy",
    "gets", "scanf", "fscanf", "sscanf", "fread", "recv",
    "malloc", "calloc", "realloc", "free",
    "fopen", "open", "connect", "bind", "listen", "accept",
    "eval", "innerhtml", "document.write", "setattribute",
    "preparestatement", "createquery", "executequery", "executeupdate",
]

# Source patterns for is_source feature
SOURCE_PATTERNS = [
    "request", "getparameter", "getquerystring", "getheader",
    "getinputstream", "getreader", "getcookies", "getpathinfo",
    "input", "argv", "stdin", "environ", "getenv",
    "args", "fgets", "fread", "recv", "recvfrom",
    "read", "readline", "readlines", "scanner",
    "bufferedreader", "fileinputstream",
    "socket", "urlconnection", "httpservletrequest",
]

print("Configuration loaded.")
print(f"  Model: MiniGAT ({CONFIG['input_dim']} -> {CONFIG['hidden_dim']} -> {CONFIG['output_dim']})")
print(f"  Heads: L1={CONFIG['num_heads_l1']}, L2={CONFIG['num_heads_l2']}")
print(f"  Target CWEs: {len(CONFIG['target_cwes'])}")
print(f"  Conformal alpha: {CONFIG['alpha']}")

## Cell 3: Download Juliet Test Suite

In [ ]:
# ============================================================================
# Cell 3: Download and Extract Juliet Test Suite
# ============================================================================
import zipfile
import urllib.request
import re
import glob
from pathlib import Path
from collections import defaultdict, Counter

JULIET_DIR = Path("/kaggle/working/juliet")
JULIET_DIR.mkdir(exist_ok=True)

# --------------------------------------------------------------------------
# Download Juliet C/C++ and Java test suites from NIST / GitHub mirrors
# --------------------------------------------------------------------------
JULIET_URLS = {
    "c_cpp": "https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-c-cplusplus-v1-3.zip",
    "java": "https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-java-v1-3.zip",
}

# Fallback: GitHub mirrors if NIST is unreachable
JULIET_FALLBACK_URLS = {
    "c_cpp": "https://github.com/compositionalenterprises/juliet-test-suite-c-cplusplus/archive/refs/heads/master.zip",
    "java": "https://github.com/compositionalenterprises/juliet-test-suite-java/archive/refs/heads/master.zip",
}


def download_with_fallback(name, primary_url, fallback_url, dest_dir):
    """Try primary URL, fall back to secondary."""
    zip_path = dest_dir / f"{name}.zip"
    if zip_path.exists():
        print(f"  {name}: already downloaded")
        return zip_path

    for url_label, url in [("primary", primary_url), ("fallback", fallback_url)]:
        try:
            print(f"  {name}: downloading from {url_label}...")
            urllib.request.urlretrieve(url, str(zip_path))
            print(f"  {name}: downloaded ({zip_path.stat().st_size / 1e6:.1f} MB)")
            return zip_path
        except Exception as e:
            print(f"  {name}: {url_label} failed: {e}")
            if zip_path.exists():
                zip_path.unlink()

    print(f"  WARNING: Could not download {name}. Will generate synthetic data.")
    return None


print("Downloading Juliet Test Suite...")
zip_paths = {}
for name in JULIET_URLS:
    zp = download_with_fallback(
        name, JULIET_URLS[name], JULIET_FALLBACK_URLS[name], JULIET_DIR
    )
    if zp is not None:
        zip_paths[name] = zp

# --------------------------------------------------------------------------
# Extract
# --------------------------------------------------------------------------
for name, zp in zip_paths.items():
    extract_dir = JULIET_DIR / name
    if extract_dir.exists() and any(extract_dir.iterdir()):
        print(f"  {name}: already extracted")
        continue
    print(f"  Extracting {name}...")
    try:
        with zipfile.ZipFile(str(zp), "r") as zf:
            zf.extractall(str(extract_dir))
        print(f"  {name}: extracted")
    except Exception as e:
        print(f"  {name}: extraction failed: {e}")

# --------------------------------------------------------------------------
# Discover and classify test case files
# --------------------------------------------------------------------------
# Juliet naming convention:
#   CWE<num>_<description>__<variant>.java  (or .c / .cpp)
#   Files with "bad" in name or method -> vulnerable (label=1)
#   Files with "good" in name or method -> safe (label=0)
#   s01/s02/... are sub-flow variants

CWE_PATTERN = re.compile(r"CWE(\d+)")

test_cases = []  # list of dict: {path, cwe, label, language}

def classify_juliet_file(filepath):
    """Classify a Juliet file as bad (vulnerable) or good (safe)."""
    fname = Path(filepath).stem.lower()
    # Explicit bad/good markers
    if "_bad" in fname or fname.endswith("bad"):
        return 1  # vulnerable
    if "_good" in fname or fname.endswith("good"):
        return 0  # safe
    # Check variant markers: a=bad, b=good in many Juliet patterns
    if "__a" in fname and "__b" not in fname:
        return 1
    if "__b" in fname:
        return 0
    # Read first few lines to check for bad/good markers
    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            header = f.read(2000).lower()
        if "bad(" in header or "badsource" in header or "badsink" in header:
            return 1
        if "good(" in header or "goodsource" in header or "goodsink" in header:
            return 0
        # If file name contains just the CWE with no marker, check content
        if "bad" in header and "good" not in header:
            return 1
        if "good" in header:
            return 0
    except Exception:
        pass
    return -1  # unknown


def scan_juliet_directory(base_dir, language, extensions):
    """Scan a Juliet directory tree for test case files."""
    found = []
    base = Path(base_dir)
    if not base.exists():
        return found
    for ext in extensions:
        for fpath in base.rglob(f"*{ext}"):
            fname = fpath.name
            # Skip support/helper files
            if any(skip in fname.lower() for skip in [
                "testcasesupport", "helper", "abstract", "main.",
                "io.", "cwe_", "_base"
            ]):
                continue
            # Must match CWE pattern
            m = CWE_PATTERN.search(str(fpath))
            if not m:
                continue
            cwe_num = m.group(1)
            cwe_id = f"CWE-{cwe_num}"
            # Only keep target CWEs
            if cwe_id not in CONFIG["target_cwes"]:
                continue
            label = classify_juliet_file(str(fpath))
            if label < 0:
                continue  # skip ambiguous files
            found.append({
                "path": str(fpath),
                "cwe": cwe_id,
                "label": label,
                "language": language,
                "size": fpath.stat().st_size,
            })
    return found


print("\nScanning for test cases...")
# Scan C/C++
for d in (JULIET_DIR / "c_cpp").iterdir() if (JULIET_DIR / "c_cpp").exists() else []:
    test_cases.extend(scan_juliet_directory(d, "c_cpp", [".c", ".cpp"]))
# Scan Java
for d in (JULIET_DIR / "java").iterdir() if (JULIET_DIR / "java").exists() else []:
    test_cases.extend(scan_juliet_directory(d, "java", [".java"]))

print(f"  Found {len(test_cases)} test case files")

# --------------------------------------------------------------------------
# If download/extraction failed, generate synthetic Juliet-like data
# --------------------------------------------------------------------------
USE_SYNTHETIC = len(test_cases) < 100

if USE_SYNTHETIC:
    print("\n  Juliet download unavailable. Generating synthetic dataset...")
    import random
    random.seed(42)

    # Templates mimicking real Juliet CWEs
    VULN_TEMPLATES = {
        "CWE-89": [
            'import java.sql.*;\npublic class CWE89_Bad_{i} {{\n    public void bad(String data) {{\n        Connection conn = DriverManager.getConnection("db");\n        Statement stmt = conn.createStatement();\n        stmt.executeQuery("SELECT * FROM t WHERE id=\'" + data + "\'");\n    }}\n}}',
            '#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\nvoid bad_{i}(char *input) {{\n    char query[512];\n    sprintf(query, "SELECT * FROM users WHERE name=\'%s\'", input);\n    execute_query(query);\n}}',
        ],
        "CWE-78": [
            'import java.io.*;\npublic class CWE78_Bad_{i} {{\n    public void bad(String cmd) {{\n        Runtime rt = Runtime.getRuntime();\n        rt.exec("ls " + cmd);\n    }}\n}}',
            '#include <stdlib.h>\nvoid bad_{i}(char *input) {{\n    char cmd[256];\n    sprintf(cmd, "ping %s", input);\n    system(cmd);\n}}',
        ],
        "CWE-79": [
            'import javax.servlet.http.*;\npublic class CWE79_Bad_{i} extends HttpServlet {{\n    public void doGet(HttpServletRequest req, HttpServletResponse resp) {{\n        String name = req.getParameter("name");\n        resp.getWriter().println("<h1>" + name + "</h1>");\n    }}\n}}',
        ],
        "CWE-476": [
            '#include <stdio.h>\nvoid bad_{i}(int *ptr) {{\n    int *p = NULL;\n    if (rand() % 2 == 0) {{ p = ptr; }}\n    printf("%d", *p);\n}}',
        ],
        "CWE-190": [
            '#include <limits.h>\nint bad_{i}(int a) {{\n    int result = a + INT_MAX;\n    return result;\n}}',
        ],
        "CWE-401": [
            '#include <stdlib.h>\nvoid bad_{i}() {{\n    char *buf = (char*)malloc(100);\n    buf[0] = \'A\';\n    /* no free */\n}}',
        ],
        "CWE-122": [
            '#include <string.h>\n#include <stdlib.h>\nvoid bad_{i}(char *src) {{\n    char *dst = (char*)malloc(10);\n    strcpy(dst, src);  /* heap overflow */\n    free(dst);\n}}',
        ],
        "CWE-121": [
            '#include <string.h>\nvoid bad_{i}(char *input) {{\n    char buf[16];\n    strcpy(buf, input);  /* stack overflow */\n}}',
        ],
        "CWE-134": [
            '#include <stdio.h>\nvoid bad_{i}(char *input) {{\n    printf(input);  /* format string vuln */\n}}',
        ],
        "CWE-369": [
            '#include <stdio.h>\nint bad_{i}(int a, int b) {{\n    return a / b;  /* potential divide by zero */\n}}',
        ],
        "CWE-690": [
            '#include <stdlib.h>\nvoid bad_{i}() {{\n    char *p = (char*)malloc(100);\n    p[0] = \'x\';  /* no null check */\n}}',
        ],
        "CWE-80": [
            'import javax.servlet.http.*;\npublic class CWE80_Bad_{i} extends HttpServlet {{\n    public void doGet(HttpServletRequest req, HttpServletResponse resp) {{\n        resp.getWriter().write(req.getParameter("data"));\n    }}\n}}',
        ],
        "CWE-36": [
            'import java.io.*;\npublic class CWE36_Bad_{i} {{\n    public void bad(String path) {{\n        File f = new File(path);\n        f.delete();\n    }}\n}}',
        ],
        "CWE-400": [
            'import java.util.*;\npublic class CWE400_Bad_{i} {{\n    public void bad(int size) {{\n        List<byte[]> list = new ArrayList<>();\n        for (int i=0; i<size; i++) list.add(new byte[1024*1024]);\n    }}\n}}',
        ],
        "CWE-197": [
            '#include <stdio.h>\nvoid bad_{i}(long val) {{\n    short s = (short)val;  /* truncation */\n    printf("%d", s);\n}}',
        ],
    }

    SAFE_TEMPLATES = {
        "CWE-89": [
            'import java.sql.*;\npublic class CWE89_Good_{i} {{\n    public void good(String data) {{\n        Connection conn = DriverManager.getConnection("db");\n        PreparedStatement ps = conn.prepareStatement("SELECT * FROM t WHERE id=?");\n        ps.setString(1, data);\n        ps.executeQuery();\n    }}\n}}',
            '#include <stdio.h>\n#include <sqlite3.h>\nvoid good_{i}(const char *input) {{\n    sqlite3_stmt *stmt;\n    sqlite3_prepare_v2(db, "SELECT * FROM users WHERE name=?", -1, &stmt, NULL);\n    sqlite3_bind_text(stmt, 1, input, -1, SQLITE_STATIC);\n    sqlite3_step(stmt);\n}}',
        ],
        "CWE-78": [
            'import java.io.*;\npublic class CWE78_Good_{i} {{\n    public void good() {{\n        ProcessBuilder pb = new ProcessBuilder("ls", "-la");\n        pb.start();\n    }}\n}}',
            '#include <stdlib.h>\n#include <string.h>\nvoid good_{i}() {{\n    const char *cmd = "ping 127.0.0.1";\n    system(cmd);  /* hardcoded, no user input */\n}}',
        ],
        "CWE-79": [
            'import javax.servlet.http.*;\nimport org.owasp.encoder.Encode;\npublic class CWE79_Good_{i} extends HttpServlet {{\n    public void doGet(HttpServletRequest req, HttpServletResponse resp) {{\n        String name = Encode.forHtml(req.getParameter("name"));\n        resp.getWriter().println("<h1>" + name + "</h1>");\n    }}\n}}',
        ],
        "CWE-476": [
            '#include <stdio.h>\nvoid good_{i}(int *ptr) {{\n    if (ptr != NULL) {{\n        printf("%d", *ptr);\n    }}\n}}',
        ],
        "CWE-190": [
            '#include <limits.h>\n#include <stdio.h>\nint good_{i}(int a) {{\n    if (a > 0 && a < INT_MAX - 100) {{\n        return a + 100;\n    }}\n    return 0;\n}}',
        ],
        "CWE-401": [
            '#include <stdlib.h>\nvoid good_{i}() {{\n    char *buf = (char*)malloc(100);\n    buf[0] = \'A\';\n    free(buf);\n}}',
        ],
        "CWE-122": [
            '#include <string.h>\n#include <stdlib.h>\nvoid good_{i}(char *src) {{\n    size_t len = strlen(src);\n    char *dst = (char*)malloc(len + 1);\n    strncpy(dst, src, len + 1);\n    free(dst);\n}}',
        ],
        "CWE-121": [
            '#include <string.h>\nvoid good_{i}(char *input) {{\n    char buf[256];\n    strncpy(buf, input, sizeof(buf) - 1);\n    buf[sizeof(buf) - 1] = \'\\0\';\n}}',
        ],
        "CWE-134": [
            '#include <stdio.h>\nvoid good_{i}(char *input) {{\n    printf("%s", input);  /* safe format string */\n}}',
        ],
        "CWE-369": [
            '#include <stdio.h>\nint good_{i}(int a, int b) {{\n    if (b != 0) {{ return a / b; }}\n    return 0;\n}}',
        ],
        "CWE-690": [
            '#include <stdlib.h>\nvoid good_{i}() {{\n    char *p = (char*)malloc(100);\n    if (p != NULL) {{ p[0] = \'x\'; free(p); }}\n}}',
        ],
        "CWE-80": [
            'import javax.servlet.http.*;\nimport org.owasp.encoder.Encode;\npublic class CWE80_Good_{i} extends HttpServlet {{\n    public void doGet(HttpServletRequest req, HttpServletResponse resp) {{\n        resp.getWriter().write(Encode.forHtml(req.getParameter("data")));\n    }}\n}}',
        ],
        "CWE-36": [
            'import java.io.*;\npublic class CWE36_Good_{i} {{\n    public void good(String path) {{\n        File base = new File("/safe/dir");\n        File f = new File(base, path).getCanonicalFile();\n        if (f.getPath().startsWith(base.getCanonicalPath())) f.delete();\n    }}\n}}',
        ],
        "CWE-400": [
            'import java.util.*;\npublic class CWE400_Good_{i} {{\n    private static final int MAX = 100;\n    public void good(int size) {{\n        int safe = Math.min(size, MAX);\n        List<byte[]> list = new ArrayList<>();\n        for (int i=0; i<safe; i++) list.add(new byte[1024]);\n    }}\n}}',
        ],
        "CWE-197": [
            '#include <stdio.h>\n#include <limits.h>\nvoid good_{i}(long val) {{\n    if (val >= SHRT_MIN && val <= SHRT_MAX) {{\n        short s = (short)val;\n        printf("%d", s);\n    }}\n}}',
        ],
    }

    SYNTH_DIR = Path("/kaggle/working/juliet_synthetic")
    SYNTH_DIR.mkdir(exist_ok=True)

    samples_per_cwe = 80  # ~80 bad + 80 good per CWE = ~2400 total
    test_cases = []

    for cwe in CONFIG["target_cwes"]:
        vuln_tmpls = VULN_TEMPLATES.get(cwe, VULN_TEMPLATES["CWE-89"])
        safe_tmpls = SAFE_TEMPLATES.get(cwe, SAFE_TEMPLATES["CWE-89"])

        for i in range(samples_per_cwe):
            # Bad sample
            tmpl = vuln_tmpls[i % len(vuln_tmpls)]
            code = tmpl.format(i=i)
            # Add variation
            for _ in range(random.randint(0, 3)):
                code += f"\n// variant line {random.randint(1,999)}"
            lang = "java" if "class " in code else "c_cpp"
            ext = ".java" if lang == "java" else ".c"
            fpath = SYNTH_DIR / f"{cwe}_bad_{i}{ext}"
            fpath.write_text(code, encoding="utf-8")
            test_cases.append({
                "path": str(fpath), "cwe": cwe, "label": 1,
                "language": lang, "size": len(code),
            })

            # Good sample
            tmpl = safe_tmpls[i % len(safe_tmpls)]
            code = tmpl.format(i=i)
            for _ in range(random.randint(0, 3)):
                code += f"\n// variant line {random.randint(1,999)}"
            lang = "java" if "class " in code else "c_cpp"
            ext = ".java" if lang == "java" else ".c"
            fpath = SYNTH_DIR / f"{cwe}_good_{i}{ext}"
            fpath.write_text(code, encoding="utf-8")
            test_cases.append({
                "path": str(fpath), "cwe": cwe, "label": 0,
                "language": lang, "size": len(code),
            })

    print(f"  Generated {len(test_cases)} synthetic test cases")

# --------------------------------------------------------------------------
# Dataset statistics
# --------------------------------------------------------------------------
n_bad = sum(1 for tc in test_cases if tc["label"] == 1)
n_good = sum(1 for tc in test_cases if tc["label"] == 0)
cwe_counts = Counter(tc["cwe"] for tc in test_cases)
lang_counts = Counter(tc["language"] for tc in test_cases)
sizes = [tc["size"] for tc in test_cases]

print(f"\n{'='*60}")
print(f"  Dataset Summary")
print(f"{'='*60}")
print(f"  Total files:     {len(test_cases)}")
print(f"  Vulnerable (bad): {n_bad}")
print(f"  Safe (good):      {n_good}")
print(f"  Balance ratio:    {n_bad/max(n_good,1):.2f}")
print(f"  Avg file size:    {sum(sizes)/max(len(sizes),1):.0f} bytes")
print(f"  Languages:        {dict(lang_counts)}")
print(f"  CWEs:             {len(cwe_counts)}")
print(f"  {'Synthetic' if USE_SYNTHETIC else 'Real Juliet'} data")

## Cell 4: Exploratory Data Analysis

In [ ]:
# ============================================================================
# Cell 4: EDA (Exploratory Data Analysis)
# ============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Plot 1: Distribution by CWE (bar chart) ---
ax1 = axes[0, 0]
cwes_sorted = sorted(cwe_counts.items(), key=lambda x: x[1], reverse=True)
cwe_names = [c[0] for c in cwes_sorted]
cwe_vals = [c[1] for c in cwes_sorted]
colors = plt.cm.Set3(np.linspace(0, 1, len(cwe_names)))
ax1.barh(cwe_names, cwe_vals, color=colors)
ax1.set_xlabel("Sample Count")
ax1.set_title("Distribution by CWE")
ax1.invert_yaxis()
for i, v in enumerate(cwe_vals):
    ax1.text(v + 1, i, str(v), va="center", fontsize=8)

# --- Plot 2: Class Balance ---
ax2 = axes[0, 1]
labels_pie = ["Vulnerable (bad)", "Safe (good)"]
sizes_pie = [n_bad, n_good]
ax2.pie(sizes_pie, labels=labels_pie, autopct="%1.1f%%",
        colors=["#ff6b6b", "#51cf66"], startangle=90)
ax2.set_title("Class Balance")

# --- Plot 3: Language Distribution ---
ax3 = axes[1, 0]
lang_names = list(lang_counts.keys())
lang_vals = list(lang_counts.values())
ax3.bar(lang_names, lang_vals, color=["#4dabf7", "#ffa94d"][:len(lang_names)])
ax3.set_xlabel("Language")
ax3.set_ylabel("Count")
ax3.set_title("Language Distribution")
for i, v in enumerate(lang_vals):
    ax3.text(i, v + 2, str(v), ha="center")

# --- Plot 4: File Size Distribution ---
ax4 = axes[1, 1]
ax4.hist(sizes, bins=30, color="#845ef7", alpha=0.7, edgecolor="black")
ax4.set_xlabel("File Size (bytes)")
ax4.set_ylabel("Count")
ax4.set_title("File Size Distribution")
ax4.axvline(np.mean(sizes), color="red", linestyle="--", label=f"Mean={np.mean(sizes):.0f}")
ax4.legend()

plt.suptitle("NIST Juliet Test Suite - Dataset Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/eda_analysis.png", dpi=150)
plt.show()

# Top 10 CWEs table
print(f"\n{'CWE':<12} {'Count':>6} {'Bad':>5} {'Good':>5}")
print("-" * 30)
for cwe, count in cwes_sorted[:10]:
    bad_c = sum(1 for tc in test_cases if tc["cwe"] == cwe and tc["label"] == 1)
    good_c = sum(1 for tc in test_cases if tc["cwe"] == cwe and tc["label"] == 0)
    print(f"{cwe:<12} {count:>6} {bad_c:>5} {good_c:>5}")

## Cell 5: Install Joern (with tree-sitter fallback)

In [ ]:
# ============================================================================
# Cell 5: Install Joern on Kaggle (with fallback to tree-sitter-based graphs)
# ============================================================================
import shutil

JOERN_AVAILABLE = False
JOERN_BIN = None

# Check Java first
print("Checking Java...")
try:
    result = subprocess.run(["java", "-version"], capture_output=True, text=True)
    java_version = result.stderr.split("\n")[0] if result.stderr else "unknown"
    print(f"  Java: {java_version}")
except Exception as e:
    print(f"  Java not found: {e}")

# Try to install Joern
print("\nAttempting Joern installation...")
try:
    # Download Joern install script
    subprocess.run(
        ["curl", "-sL",
         "https://github.com/joernio/joern/releases/latest/download/joern-install.sh",
         "-o", "/tmp/joern-install.sh"],
        check=True, timeout=60
    )
    # Install Joern
    result = subprocess.run(
        ["bash", "/tmp/joern-install.sh", "--install-dir=/opt/joern"],
        capture_output=True, text=True, timeout=300
    )
    if result.returncode == 0:
        joern_path = shutil.which("joern", path="/opt/joern/joern-cli/bin")
        if joern_path:
            JOERN_AVAILABLE = True
            JOERN_BIN = joern_path
            os.environ["PATH"] = f"/opt/joern/joern-cli/bin:{os.environ['PATH']}"
            print(f"  Joern installed: {joern_path}")
        else:
            print("  Joern install script succeeded but binary not found.")
    else:
        print(f"  Joern install failed (rc={result.returncode})")
        if result.stderr:
            print(f"  stderr: {result.stderr[:200]}")
except subprocess.TimeoutExpired:
    print("  Joern install timed out.")
except Exception as e:
    print(f"  Joern install error: {e}")

if not JOERN_AVAILABLE:
    print("\n  FALLBACK: Using regex-based AST graph construction.")
    print("  (Weaker than Joern CPGs but allows training to proceed.)")
else:
    print("\n  Joern is available for full CPG generation.")

## Cell 6: Build Code Property Graphs

In [ ]:
# ============================================================================
# Cell 6: Build CPGs (Code Property Graphs)
# ============================================================================
import networkx as nx
import tempfile
import time as time_module

# --------------------------------------------------------------------------
# Fallback: Regex-based graph builder (when Joern is unavailable)
# --------------------------------------------------------------------------
# Patterns for C/C++ and Java statement/construct recognition
_C_KEYWORDS = {"if", "else", "for", "while", "do", "switch", "case",
               "return", "break", "continue", "goto"}
_JAVA_KEYWORDS = _C_KEYWORDS | {"try", "catch", "finally", "throw", "throws"}
_CALL_PATTERN = re.compile(r'\b([a-zA-Z_][a-zA-Z0-9_]*)\s*\(')
_ASSIGN_PATTERN = re.compile(r'\b([a-zA-Z_][a-zA-Z0-9_]*)\s*=')
_VAR_PATTERN = re.compile(r'\b([a-zA-Z_][a-zA-Z0-9_]*)\b')


def build_fallback_graph(filepath, language):
    """Build an approximate CPG using regex-based parsing.

    Creates nodes for each statement/line and edges for:
    - AST: parent-child (block structure)
    - CFG: sequential flow between statements
    - DDG: approximate data dependencies via variable name matching
    """
    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            source = f.read()
    except Exception:
        return nx.DiGraph()

    lines = source.split("\n")
    G = nx.DiGraph()
    G.graph["source_path"] = filepath
    G.graph["language"] = language

    # Create nodes for each non-empty line
    node_id = 0
    line_to_node = {}
    indent_stack = [(0, -1)]  # (indent_level, node_id) for AST parent tracking

    for line_num, line in enumerate(lines, 1):
        stripped = line.strip()
        if not stripped or stripped.startswith("//") or stripped.startswith("/*"):
            continue

        indent = len(line) - len(line.lstrip())

        # Determine node type
        node_type = "UNKNOWN"
        if any(kw + " " in stripped or kw + "(" in stripped
               for kw in ("def ", "void ", "int ", "public ", "private ",
                          "static ", "class ")):
            node_type = "METHOD"
        elif any(stripped.startswith(kw) for kw in _C_KEYWORDS | _JAVA_KEYWORDS):
            node_type = "CONTROL_STRUCTURE"
        elif _CALL_PATTERN.search(stripped):
            node_type = "CALL"
        elif "=" in stripped:
            node_type = "IDENTIFIER"
        elif stripped in ("{", "}"):
            node_type = "BLOCK"
        elif "return" in stripped:
            node_type = "RETURN"

        G.add_node(node_id, type=node_type, code=stripped,
                   lineNumber=line_num, columnNumber=indent)
        line_to_node[line_num] = node_id

        # AST edges (parent-child via indentation)
        while indent_stack and indent <= indent_stack[-1][0] and len(indent_stack) > 1:
            indent_stack.pop()
        if indent_stack:
            parent_nid = indent_stack[-1][1]
            if parent_nid >= 0:
                G.add_edge(parent_nid, node_id, type="AST")
        indent_stack.append((indent, node_id))

        # CFG edges (sequential flow)
        if node_id > 0:
            G.add_edge(node_id - 1, node_id, type="CFG")

        node_id += 1

    # DDG edges (approximate data dependencies via variable names)
    var_defs = {}  # var_name -> node_id of last definition
    nodes_list = sorted(G.nodes())
    for nid in nodes_list:
        code = G.nodes[nid].get("code", "")
        # Check for variable definitions (assignments)
        assigns = _ASSIGN_PATTERN.findall(code)
        # Check for variable uses
        all_vars = set(_VAR_PATTERN.findall(code))
        # Add DDG edges for used variables
        for var in all_vars - set(assigns):
            if var in var_defs and var_defs[var] != nid:
                G.add_edge(var_defs[var], nid, type="DDG")
        # Update definitions
        for var in assigns:
            var_defs[var] = nid

    return G


def build_joern_graph(filepath, language):
    """Build CPG using Joern CLI."""
    with tempfile.TemporaryDirectory(prefix="cpg_") as tmpdir:
        graphml_path = os.path.join(tmpdir, "cpg.graphml")
        joern_lang = {"java": "javasrc", "c_cpp": "c"}.get(language, "c")
        export_script = (
            '@main def main(inputPath: String, outputPath: String): Unit = {\n'
            '  importCode(inputPath)\n'
            '  cpg.graph.export(outputPath, "graphml")\n'
            '}'
        )
        script_path = os.path.join(tmpdir, "export.sc")
        with open(script_path, "w") as f:
            f.write(export_script)

        cmd = [
            JOERN_BIN, "--script", script_path,
            "--param", f"inputPath={filepath}",
            "--param", f"outputPath={graphml_path}",
        ]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if result.returncode == 0 and os.path.isfile(graphml_path):
                G = nx.read_graphml(graphml_path)
                # Normalize node attributes
                norm_G = nx.DiGraph()
                for nid, attrs in G.nodes(data=True):
                    norm_G.add_node(nid,
                        type=attrs.get("label", attrs.get("type", "UNKNOWN")),
                        code=attrs.get("CODE", attrs.get("code", "")),
                        lineNumber=int(attrs.get("LINE_NUMBER",
                                                 attrs.get("lineNumber", 0))),
                    )
                for src, dst, attrs in G.edges(data=True):
                    edge_type = attrs.get("label", attrs.get("type", "UNKNOWN"))
                    norm_G.add_edge(src, dst, type=edge_type)
                return norm_G
        except Exception:
            pass
    # Fallback
    return build_fallback_graph(filepath, language)


# --------------------------------------------------------------------------
# Build graphs for all test cases
# --------------------------------------------------------------------------
print(f"Building graphs for {len(test_cases)} test cases...")
print(f"  Method: {'Joern CPG' if JOERN_AVAILABLE else 'Regex-based fallback'}")

graph_data = []  # list of (nx.DiGraph, label, cwe, language)
build_errors = 0
t0 = time_module.time()

for idx, tc in enumerate(test_cases):
    try:
        if JOERN_AVAILABLE:
            G = build_joern_graph(tc["path"], tc["language"])
        else:
            G = build_fallback_graph(tc["path"], tc["language"])

        if G.number_of_nodes() >= 3:  # skip trivial graphs
            graph_data.append((G, tc["label"], tc["cwe"], tc["language"]))
    except Exception as e:
        build_errors += 1

    if (idx + 1) % 200 == 0 or idx == len(test_cases) - 1:
        elapsed = time_module.time() - t0
        print(f"  [{idx+1}/{len(test_cases)}] "
              f"{len(graph_data)} graphs built, "
              f"{build_errors} errors, "
              f"{elapsed:.1f}s elapsed")

# Statistics
avg_nodes = np.mean([G.number_of_nodes() for G, _, _, _ in graph_data]) if graph_data else 0
avg_edges = np.mean([G.number_of_edges() for G, _, _, _ in graph_data]) if graph_data else 0

print(f"\n  Graphs built: {len(graph_data)}")
print(f"  Build errors: {build_errors}")
print(f"  Avg nodes per graph: {avg_nodes:.1f}")
print(f"  Avg edges per graph: {avg_edges:.1f}")
print(f"  Build time: {time_module.time() - t0:.1f}s")

## Cell 7: Build PyG Dataset with GraphCodeBERT Embeddings

In [ ]:
# ============================================================================
# Cell 7: Build PyG Dataset
# ============================================================================
from torch_geometric.data import Data
from collections import deque
from transformers import AutoModel, AutoTokenizer

# --------------------------------------------------------------------------
# Load GraphCodeBERT for node embeddings
# --------------------------------------------------------------------------
print("Loading GraphCodeBERT...")
gcb_tokenizer = AutoTokenizer.from_pretrained(CONFIG["embedding_model"])
gcb_model = AutoModel.from_pretrained(CONFIG["embedding_model"]).to(device)
gcb_model.eval()
print(f"  GraphCodeBERT loaded on {device}")


@torch.no_grad()
def embed_code_batch(code_snippets, batch_size=64):
    """Embed a batch of code snippets using GraphCodeBERT.

    Returns a tensor of shape (len(code_snippets), 768).
    """
    all_embeddings = []
    for start in range(0, len(code_snippets), batch_size):
        batch = code_snippets[start:start + batch_size]
        # Clean snippets
        batch = [s if s.strip() else "pass" for s in batch]
        encoded = gcb_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=CONFIG["max_tokens_per_node"],
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = gcb_model(**encoded)
        cls_emb = outputs.last_hidden_state[:, 0, :].cpu()  # CLS token
        all_embeddings.append(cls_emb)
    return torch.cat(all_embeddings, dim=0)


def truncate_graph_bfs(G, max_nodes):
    """Truncate a graph to max_nodes using BFS from the highest-degree node."""
    if G.number_of_nodes() <= max_nodes:
        return G

    # Start BFS from the node with highest total degree
    degrees = dict(G.degree())
    if not degrees:
        return G
    start_node = max(degrees, key=degrees.get)

    visited = set()
    queue = deque([start_node])
    visited.add(start_node)

    while queue and len(visited) < max_nodes:
        current = queue.popleft()
        for neighbor in list(G.successors(current)) + list(G.predecessors(current)):
            if neighbor not in visited and len(visited) < max_nodes:
                visited.add(neighbor)
                queue.append(neighbor)

    return G.subgraph(visited).copy()


def extract_node_features(G):
    """Extract 5-dim per-node structural features.

    Features:
      0: in_degree_norm   = in_degree / max_in_degree
      1: out_degree_norm  = out_degree / max_out_degree
      2: is_sink          = 1.0 if out_degree==0 AND code contains sink patterns
      3: is_source        = 1.0 if code contains source patterns
      4: depth_norm       = BFS depth from root / max_depth
    """
    nodes = list(G.nodes())
    n = len(nodes)
    if n == 0:
        return torch.zeros(0, 5)

    node_to_idx = {node: i for i, node in enumerate(nodes)}
    feats = torch.zeros(n, 5)

    # In-degree and out-degree
    in_degrees = [G.in_degree(nd) for nd in nodes]
    out_degrees = [G.out_degree(nd) for nd in nodes]
    max_in = max(in_degrees) if in_degrees else 1
    max_out = max(out_degrees) if out_degrees else 1
    max_in = max(max_in, 1)
    max_out = max(max_out, 1)

    for i, nd in enumerate(nodes):
        feats[i, 0] = in_degrees[i] / max_in
        feats[i, 1] = out_degrees[i] / max_out

        code = G.nodes[nd].get("code", "").lower()

        # is_sink: out_degree == 0 AND code contains known sink patterns
        if out_degrees[i] == 0 and any(p in code for p in SINK_PATTERNS):
            feats[i, 2] = 1.0
        elif any(p in code for p in SINK_PATTERNS):
            feats[i, 2] = 0.5  # partial match

        # is_source: code contains known source patterns
        if any(p in code for p in SOURCE_PATTERNS):
            feats[i, 3] = 1.0

    # BFS depth from root (first node or METHOD node)
    root = nodes[0]
    for nd in nodes:
        if G.nodes[nd].get("type") == "METHOD":
            root = nd
            break

    depths = {root: 0}
    queue = deque([root])
    while queue:
        current = queue.popleft()
        for succ in G.successors(current):
            if succ not in depths:
                depths[succ] = depths[current] + 1
                queue.append(succ)

    max_depth = max(depths.values()) if depths else 1
    max_depth = max(max_depth, 1)
    for nd in nodes:
        i = node_to_idx[nd]
        feats[i, 4] = depths.get(nd, 0) / max_depth

    return feats


def graph_to_pyg_data(G, label, cwe):
    """Convert a NetworkX DiGraph to a PyG Data object.

    Steps:
      1. Truncate to max_nodes via BFS
      2. Extract per-node structural features (5-dim)
      3. Generate GraphCodeBERT embeddings (768-dim per node)
      4. Concatenate: x = cat(embeddings, features) -> (num_nodes, 773)
      5. Build edge_index from DiGraph edges
    """
    # 1. Truncate
    G = truncate_graph_bfs(G, CONFIG["max_nodes"])
    nodes = list(G.nodes())
    n = len(nodes)
    if n < 2:
        return None

    node_to_idx = {node: i for i, node in enumerate(nodes)}

    # 2. Extract structural features
    struct_feats = extract_node_features(G)  # (n, 5)

    # 3. GraphCodeBERT embeddings
    code_snippets = []
    for nd in nodes:
        code = G.nodes[nd].get("code", "")
        if not code or not code.strip():
            code = G.nodes[nd].get("type", "UNKNOWN")
        code_snippets.append(code)

    embeddings = embed_code_batch(code_snippets,
                                  batch_size=CONFIG["embedding_batch_size"])  # (n, 768)

    # 4. Concatenate
    x = torch.cat([embeddings, struct_feats], dim=1)  # (n, 773)

    # 5. Build edge_index
    edge_list = []
    for src, dst in G.edges():
        if src in node_to_idx and dst in node_to_idx:
            edge_list.append([node_to_idx[src], node_to_idx[dst]])

    if not edge_list:
        # Add self-loops if no edges
        edge_list = [[i, i] for i in range(n)]

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    y = torch.tensor([label], dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y)
    data.cwe = cwe  # store CWE for per-CWE analysis
    return data


# --------------------------------------------------------------------------
# Build the full PyG dataset
# --------------------------------------------------------------------------
print(f"\nBuilding PyG dataset ({len(graph_data)} graphs)...")
print("  Generating GraphCodeBERT embeddings + structural features...")

pyg_dataset = []
t0 = time_module.time()

for idx, (G, label, cwe, lang) in enumerate(graph_data):
    data = graph_to_pyg_data(G, label, cwe)
    if data is not None:
        pyg_dataset.append(data)

    if (idx + 1) % 100 == 0 or idx == len(graph_data) - 1:
        elapsed = time_module.time() - t0
        rate = (idx + 1) / max(elapsed, 0.01)
        remaining = (len(graph_data) - idx - 1) / max(rate, 0.01)
        print(f"  [{idx+1}/{len(graph_data)}] "
              f"{len(pyg_dataset)} data objects, "
              f"{elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining")

print(f"\n  PyG dataset built: {len(pyg_dataset)} graphs")
print(f"  Feature dim per node: {pyg_dataset[0].x.shape[1] if pyg_dataset else 'N/A'}")
print(f"  Total build time: {time_module.time() - t0:.1f}s")

# Save dataset
torch.save(pyg_dataset, "/kaggle/working/juliet_graphs.pt")
print(f"  Saved to /kaggle/working/juliet_graphs.pt")

## Cell 8: Dataset Split

In [ ]:
# ============================================================================
# Cell 8: Stratified Dataset Split
# ============================================================================
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

# Free GPU memory from GraphCodeBERT (no longer needed)
del gcb_model
del gcb_tokenizer
if device == "cuda":
    torch.cuda.empty_cache()
print("GraphCodeBERT unloaded to free GPU memory.")

# Extract labels for stratification
all_labels = [d.y.item() for d in pyg_dataset]
indices = list(range(len(pyg_dataset)))

# 70% train, 15% validation, 15% calibration+test
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, stratify=all_labels, random_state=42
)
temp_labels = [all_labels[i] for i in temp_idx]
val_idx, cal_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_labels, random_state=42
)

train_data = [pyg_dataset[i] for i in train_idx]
val_data = [pyg_dataset[i] for i in val_idx]
cal_data = [pyg_dataset[i] for i in cal_idx]

# Create DataLoaders
train_loader = DataLoader(train_data, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_data, batch_size=CONFIG["batch_size"])
cal_loader = DataLoader(cal_data, batch_size=CONFIG["batch_size"])

# Print split statistics
def label_stats(data_list):
    labels = [d.y.item() for d in data_list]
    n0 = labels.count(0)
    n1 = labels.count(1)
    return n0, n1, len(labels)

for name, dlist in [("Train", train_data), ("Val", val_data), ("Cal+Test", cal_data)]:
    n0, n1, total = label_stats(dlist)
    print(f"  {name:>10}: {total:>5} samples  "
          f"(safe={n0}, vuln={n1}, ratio={n1/max(n0,1):.2f})")

print(f"\n  Batch size: {CONFIG['batch_size']}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Cal batches: {len(cal_loader)}")

## Cell 9: Define MiniGAT Model

In [ ]:
# ============================================================================
# Cell 9: Define MiniGAT Model (inline, no imports from src/)
# ============================================================================
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool


class MiniGAT(nn.Module):
    """SEC-C Mini-GAT: 2-layer Graph Attention Network for vulnerability detection.

    Architecture:
        Input: 773-dim (768 GraphCodeBERT + 5 structural features)
            |
        Linear 773 -> 256 + ReLU
            |
        GATConv (256 -> 256, 4 heads x 64, concat) + ReLU + Dropout
            |
        GATConv (256 -> 128, 4 heads x 32, concat) + ReLU
            |
        Global Mean Pooling -> 128-dim graph embedding
            |
        +-- Classifier: Linear 128 -> 2 (safe / vulnerable)
        +-- Confidence:  Linear 128 -> 1, Sigmoid
    """

    def __init__(self, input_dim=773, hidden_dim=256, output_dim=128,
                 num_heads_l1=4, num_heads_l2=4, dropout=0.3, num_classes=2):
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.dropout_rate = dropout

        # Projection layer: 773 -> 256
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        # GAT Layer 1: multi-head attention with concatenation
        # 4 heads x 64 = 256, so output stays at hidden_dim
        self.gat1 = GATConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim // num_heads_l1,
            heads=num_heads_l1,
            concat=True,
            dropout=dropout,
        )

        # GAT Layer 2: 4 heads x 32 = 128, output at output_dim
        self.gat2 = GATConv(
            in_channels=hidden_dim,
            out_channels=output_dim // num_heads_l2,
            heads=num_heads_l2,
            concat=True,
            dropout=dropout,
        )

        self.dropout_layer = nn.Dropout(dropout)

        # Task heads
        self.classifier = nn.Linear(output_dim, num_classes)
        self.confidence_head = nn.Sequential(
            nn.Linear(output_dim, 1),
            nn.Sigmoid(),
        )

        # Storage for attention weights (for explainability)
        self._attn_weights_l1 = None
        self._attn_weights_l2 = None

    def forward(self, x, edge_index, batch):
        """Full forward pass.

        Args:
            x: Node feature matrix (N, 773)
            edge_index: COO edge index (2, E)
            batch: Batch vector mapping nodes to graphs (N,)

        Returns:
            (class_logits, confidence): logits (B, 2), confidence (B,)
        """
        # 1) Project 773 -> 256
        h = F.relu(self.input_proj(x))

        # 2) GAT Layer 1 (256 -> 256 via 4 heads x 64)
        h, attn1 = self.gat1(h, edge_index, return_attention_weights=True)
        self._attn_weights_l1 = attn1[1]
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout_rate, training=self.training)

        # 3) GAT Layer 2 (256 -> 128 via 4 heads x 32)
        h, attn2 = self.gat2(h, edge_index, return_attention_weights=True)
        self._attn_weights_l2 = attn2[1]
        h = F.relu(h)

        # 4) Global mean pooling -> (B, 128)
        graph_emb = global_mean_pool(h, batch)

        # 5) Task heads
        logits = self.classifier(graph_emb)             # (B, 2)
        confidence = self.confidence_head(graph_emb)     # (B, 1)

        return logits, confidence.squeeze(-1)

    def parameter_count(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable}


# Instantiate the model
model = MiniGAT(
    input_dim=CONFIG["input_dim"],
    hidden_dim=CONFIG["hidden_dim"],
    output_dim=CONFIG["output_dim"],
    num_heads_l1=CONFIG["num_heads_l1"],
    num_heads_l2=CONFIG["num_heads_l2"],
    dropout=CONFIG["dropout"],
    num_classes=CONFIG["num_classes"],
).to(device)

pc = model.parameter_count()
print(f"MiniGAT created on {device}")
print(f"  Total parameters:     {pc['total']:,}")
print(f"  Trainable parameters: {pc['trainable']:,}")
print(f"\n  Architecture:")
print(f"    Input:    {CONFIG['input_dim']} (768 GCB + 5 structural)")
print(f"    Hidden:   {CONFIG['hidden_dim']} (4 heads x {CONFIG['hidden_dim']//4})")
print(f"    Output:   {CONFIG['output_dim']} (4 heads x {CONFIG['output_dim']//4})")
print(f"    Classes:  {CONFIG['num_classes']} (safe / vulnerable)")
print(f"    Dropout:  {CONFIG['dropout']}")

## Cell 10: Training Loop

In [ ]:
# ============================================================================
# Cell 10: Training Loop with Early Stopping
# ============================================================================
import copy
from torch.optim import Adam

# --------------------------------------------------------------------------
# Compute class weights for imbalanced data
# --------------------------------------------------------------------------
train_labels = [d.y.item() for d in train_data]
n_safe_train = train_labels.count(0)
n_vuln_train = train_labels.count(1)
total_train = len(train_labels)

# Inverse frequency weighting
w_safe = total_train / (2 * max(n_safe_train, 1))
w_vuln = total_train / (2 * max(n_vuln_train, 1))
class_weights = torch.tensor([w_safe, w_vuln], dtype=torch.float32).to(device)
print(f"Class weights: safe={w_safe:.3f}, vuln={w_vuln:.3f}")

# --------------------------------------------------------------------------
# Setup optimizer and loss
# --------------------------------------------------------------------------
optimizer = Adam(model.parameters(), lr=CONFIG["lr"],
                 weight_decay=CONFIG["weight_decay"])
criterion = nn.CrossEntropyLoss(weight=class_weights)


def compute_metrics(preds, labels):
    """Compute binary classification metrics."""
    n = len(preds)
    if n == 0:
        return {"accuracy": 0, "precision": 0, "recall": 0, "f1": 0}
    tp = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 1)
    fp = sum(1 for p, l in zip(preds, labels) if p == 1 and l == 0)
    fn = sum(1 for p, l in zip(preds, labels) if p == 0 and l == 1)
    correct = sum(1 for p, l in zip(preds, labels) if p == l)
    accuracy = correct / n
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


def train_one_epoch():
    """Train for one epoch. Returns average loss."""
    model.train()
    total_loss = 0.0
    num_batches = 0

    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()

        logits, confidence = model(data.x, data.edge_index, data.batch)
        labels = data.y

        # Classification loss (weighted cross-entropy)
        cls_loss = criterion(logits, labels)

        # Confidence calibration loss
        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            correct = (preds == labels).float()

        conf_loss = F.binary_cross_entropy(confidence, correct)

        loss = cls_loss + CONFIG["confidence_loss_weight"] * conf_loss

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["grad_clip"])
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(num_batches, 1)


@torch.no_grad()
def validate():
    """Validate on the validation set. Returns (loss, metrics_dict)."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    all_preds = []
    all_labels = []

    for data in val_loader:
        data = data.to(device)
        logits, confidence = model(data.x, data.edge_index, data.batch)
        labels = data.y

        cls_loss = criterion(logits, labels)
        preds = logits.argmax(dim=-1)
        correct = (preds == labels).float()
        conf_loss = F.binary_cross_entropy(confidence, correct)
        loss = cls_loss + CONFIG["confidence_loss_weight"] * conf_loss

        total_loss += loss.item()
        num_batches += 1
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / max(num_batches, 1)
    metrics = compute_metrics(all_preds, all_labels)
    return avg_loss, metrics


# --------------------------------------------------------------------------
# Training loop
# --------------------------------------------------------------------------
history = {
    "train_loss": [], "val_loss": [],
    "val_acc": [], "val_f1": [],
    "val_precision": [], "val_recall": [],
}
best_val_loss = float("inf")
best_state_dict = None
best_epoch = 0
patience_counter = 0
training_start = time_module.time()

EPOCHS = CONFIG["epochs"]
PATIENCE = CONFIG["patience"]

print(f"\n{'='*70}")
print(f"  Training MiniGAT for up to {EPOCHS} epochs on {device}")
print(f"  LR={CONFIG['lr']}, WD={CONFIG['weight_decay']}, Patience={PATIENCE}")
print(f"{'='*70}")
print(f"{'Epoch':>6} {'TrLoss':>8} {'VaLoss':>8} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Time':>6}")
print("-" * 70)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time_module.time()

    train_loss = train_one_epoch()
    val_loss, val_metrics = validate()

    epoch_time = time_module.time() - epoch_start

    # Record history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])

    # Print every epoch
    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state_dict = copy.deepcopy(model.state_dict())
        patience_counter = 0
        marker = " *"
    else:
        patience_counter += 1

    if epoch <= 5 or epoch % 5 == 0 or epoch == EPOCHS or marker:
        print(f"{epoch:>5d}  {train_loss:>8.4f} {val_loss:>8.4f} "
              f"{val_metrics['accuracy']:>6.3f} {val_metrics['precision']:>6.3f} "
              f"{val_metrics['recall']:>6.3f} {val_metrics['f1']:>6.3f} "
              f"{epoch_time:>5.1f}s{marker}")

    if patience_counter >= PATIENCE:
        print(f"\n  Early stopping at epoch {epoch} (best: epoch {best_epoch})")
        break

# Restore best model
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

training_time = time_module.time() - training_start
print(f"\n  Training complete in {training_time:.1f}s ({training_time/60:.1f} min)")
print(f"  Best epoch: {best_epoch}, best val_loss: {best_val_loss:.4f}")

# --------------------------------------------------------------------------
# Plot training curves
# --------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

# Loss curve
ax1.plot(epochs_range, history["train_loss"], "b-", label="Train Loss", linewidth=2)
ax1.plot(epochs_range, history["val_loss"], "r-", label="Val Loss", linewidth=2)
ax1.axvline(best_epoch, color="green", linestyle="--", alpha=0.7, label=f"Best (epoch {best_epoch})")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# F1 / Accuracy curve
ax2.plot(epochs_range, history["val_f1"], "g-", label="Val F1", linewidth=2)
ax2.plot(epochs_range, history["val_acc"], "m-", label="Val Accuracy", linewidth=2)
ax2.plot(epochs_range, history["val_precision"], "c--", label="Val Precision", alpha=0.7)
ax2.plot(epochs_range, history["val_recall"], "y--", label="Val Recall", alpha=0.7)
ax2.axvline(best_epoch, color="green", linestyle="--", alpha=0.7)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Validation Metrics")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.suptitle("MiniGAT Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150)
plt.show()

## Cell 11: Evaluation

In [ ]:
# ============================================================================
# Cell 11: Evaluation on Test/Calibration Set
# ============================================================================
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, roc_auc_score
)

model.eval()

all_preds = []
all_labels = []
all_probs = []
all_cwes = []

with torch.no_grad():
    for data in cal_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        probs = F.softmax(logits, dim=-1)

        preds = logits.argmax(dim=-1).cpu().tolist()
        labels = data.y.cpu().tolist()

        all_preds.extend(preds)
        all_labels.extend(labels)
        all_probs.extend(probs.cpu().numpy())

        # Collect CWEs for per-CWE analysis (robust across PyG versions)
        if hasattr(data, "cwe"):
            cwes = data.cwe
            if isinstance(cwes, str):
                cwes = [cwes] * len(labels)
            elif not isinstance(cwes, (list, tuple)):
                try:
                    cwes = list(cwes)
                except TypeError:
                    cwes = [str(cwes)] * len(labels)
            all_cwes.extend(cwes)

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Classification metrics
test_metrics = compute_metrics(all_preds.tolist(), all_labels.tolist())

# AUC-ROC
try:
    auc_roc = roc_auc_score(all_labels, all_probs[:, 1])
except Exception:
    auc_roc = 0.0

print(f"{'='*60}")
print(f"  Evaluation Results")
print(f"{'='*60}")
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  F1 Score:  {test_metrics['f1']:.4f}")
print(f"  AUC-ROC:   {auc_roc:.4f}")

# Classification report
print(f"\n  Classification Report:")
print(classification_report(all_labels, all_preds,
                            target_names=["safe", "vulnerable"]))

# --------------------------------------------------------------------------
# Plot confusion matrix and ROC curve
# --------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
im = ax1.imshow(cm, interpolation="nearest", cmap="Blues")
ax1.set_title("Confusion Matrix")
ax1.set_ylabel("True Label")
ax1.set_xlabel("Predicted Label")
tick_marks = [0, 1]
ax1.set_xticks(tick_marks)
ax1.set_xticklabels(["Safe", "Vulnerable"])
ax1.set_yticks(tick_marks)
ax1.set_yticklabels(["Safe", "Vulnerable"])
# Add text annotations
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax1.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color=color, fontsize=16, fontweight="bold")
plt.colorbar(im, ax=ax1)

# ROC curve
try:
    fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
    ax2.plot(fpr, tpr, "b-", linewidth=2, label=f"MiniGAT (AUC={auc_roc:.3f})")
    ax2.plot([0, 1], [0, 1], "r--", alpha=0.5, label="Random")
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title("ROC Curve")
    ax2.legend(loc="lower right")
    ax2.grid(True, alpha=0.3)
except Exception:
    ax2.text(0.5, 0.5, "ROC not available", ha="center", va="center")

plt.tight_layout()
plt.savefig("/kaggle/working/evaluation_results.png", dpi=150)
plt.show()

# --------------------------------------------------------------------------
# Per-CWE breakdown
# --------------------------------------------------------------------------
if all_cwes:
    print(f"\n  Per-CWE Breakdown:")
    print(f"  {'CWE':<12} {'Count':>6} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
    print("  " + "-" * 50)

    cwe_set = sorted(set(all_cwes))
    for cwe in cwe_set:
        mask = np.array([c == cwe for c in all_cwes])
        if mask.sum() < 2:
            continue
        cwe_preds = all_preds[mask]
        cwe_labels = all_labels[mask]
        m = compute_metrics(cwe_preds.tolist(), cwe_labels.tolist())
        print(f"  {cwe:<12} {mask.sum():>6} {m['accuracy']:>6.3f} "
              f"{m['precision']:>6.3f} {m['recall']:>6.3f} {m['f1']:>6.3f}")

## Cell 12: Conformal Prediction Calibration

In [ ]:
# ============================================================================
# Cell 12: Conformal Prediction Calibration (APS)
# ============================================================================
import math

CLASS_LABELS = ["safe", "vulnerable"]


def compute_aps_nonconformity(softmax_probs, true_labels):
    """Compute APS nonconformity scores for calibration.

    For each sample:
    1. Sort classes by descending softmax probability.
    2. Compute cumulative sum of sorted probabilities.
    3. The nonconformity score is the cumulative sum at the position
       where the true label first appears (inclusive).

    Higher score = model was less certain about the true class.
    """
    n = len(true_labels)
    scores = np.zeros(n, dtype=np.float64)

    for i in range(n):
        probs = softmax_probs[i]
        true_label = true_labels[i]

        sorted_indices = np.argsort(-probs)
        sorted_probs = probs[sorted_indices]
        cumsum = np.cumsum(sorted_probs)

        rank = int(np.where(sorted_indices == true_label)[0][0])
        scores[i] = cumsum[rank]

    return scores


def build_prediction_set(probs, threshold):
    """Build conformal prediction set for a single sample.

    Greedily includes classes in descending probability order until
    cumulative probability >= threshold.
    """
    sorted_indices = np.argsort(-probs)
    sorted_probs = probs[sorted_indices]
    cumsum = np.cumsum(sorted_probs)

    pred_set = []
    for j, idx in enumerate(sorted_indices):
        pred_set.append(CLASS_LABELS[int(idx)])
        if cumsum[j] >= threshold:
            break

    if not pred_set:
        pred_set.append(CLASS_LABELS[int(sorted_indices[0])])

    return pred_set


# --------------------------------------------------------------------------
# Collect softmax probabilities on calibration set
# --------------------------------------------------------------------------
print(f"{'='*60}")
print(f"  Conformal Prediction Calibration (APS, alpha={CONFIG['alpha']})")
print(f"{'='*60}")

model.eval()
cal_softmax = []
cal_labels = []

with torch.no_grad():
    for data in cal_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        cal_softmax.append(probs)
        cal_labels.extend(data.y.cpu().tolist())

cal_softmax = np.concatenate(cal_softmax, axis=0)
cal_labels_np = np.array(cal_labels, dtype=np.int64)
n_cal = len(cal_labels_np)

# --------------------------------------------------------------------------
# Compute nonconformity scores and threshold
# --------------------------------------------------------------------------
alpha = CONFIG["alpha"]
scores = compute_aps_nonconformity(cal_softmax, cal_labels_np)

# Corrected quantile level
quantile_level = min(math.ceil((n_cal + 1) * (1.0 - alpha)) / n_cal, 1.0)
try:
    threshold = float(np.quantile(scores, quantile_level, method="higher"))
except TypeError:
    # NumPy < 1.22 uses 'interpolation' instead of 'method'
    threshold = float(np.quantile(scores, quantile_level, interpolation="higher"))

print(f"\n  Calibration samples:    {n_cal}")
print(f"  Alpha:                  {alpha}")
print(f"  Quantile level:         {quantile_level:.4f}")
print(f"  APS threshold:          {threshold:.4f}")
print(f"  Mean nonconformity:     {scores.mean():.4f}")
print(f"  Median nonconformity:   {np.median(scores):.4f}")
print(f"  Min nonconformity:      {scores.min():.4f}")
print(f"  Max nonconformity:      {scores.max():.4f}")

# --------------------------------------------------------------------------
# Verify empirical coverage
# --------------------------------------------------------------------------
covered = 0
singleton_count = 0
ambiguous_count = 0
set_sizes = []

for i in range(n_cal):
    pred_set = build_prediction_set(cal_softmax[i], threshold)
    set_sizes.append(len(pred_set))

    true_label_name = CLASS_LABELS[cal_labels_np[i]]
    if true_label_name in pred_set:
        covered += 1
    if len(pred_set) == 1:
        singleton_count += 1
    else:
        ambiguous_count += 1

empirical_coverage = covered / n_cal
singleton_rate = singleton_count / n_cal
ambiguous_rate = ambiguous_count / n_cal

print(f"\n  Empirical coverage:     {empirical_coverage:.4f}  (target: >= {1 - alpha:.4f})")
print(f"  Singleton predictions:  {singleton_count}/{n_cal} ({singleton_rate:.1%})")
print(f"  Ambiguous (-> LLM):     {ambiguous_count}/{n_cal} ({ambiguous_rate:.1%})")
print(f"  Mean set size:          {np.mean(set_sizes):.2f}")

coverage_ok = empirical_coverage >= (1 - alpha)
print(f"\n  Coverage guarantee: {'MET' if coverage_ok else 'NOT MET'}")

# --------------------------------------------------------------------------
# Plot calibration diagnostics
# --------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Score distribution
ax1 = axes[0]
ax1.hist(scores, bins=30, color="#4dabf7", alpha=0.7, edgecolor="black")
ax1.axvline(threshold, color="red", linestyle="--", linewidth=2,
            label=f"Threshold={threshold:.3f}")
ax1.set_xlabel("Nonconformity Score")
ax1.set_ylabel("Count")
ax1.set_title("APS Nonconformity Score Distribution")
ax1.legend()

# Plot 2: Prediction set size distribution
ax2 = axes[1]
set_size_counts = Counter(set_sizes)
sizes_unique = sorted(set_size_counts.keys())
counts = [set_size_counts[s] for s in sizes_unique]
bars = ax2.bar(sizes_unique, counts, color=["#51cf66", "#ffa94d"][:len(sizes_unique)])
ax2.set_xlabel("Prediction Set Size")
ax2.set_ylabel("Count")
ax2.set_title("Prediction Set Size Distribution")
ax2.set_xticks(sizes_unique)
ax2.set_xticklabels([f"{s}\n({'Singleton' if s==1 else 'Ambiguous'})" for s in sizes_unique])
for bar, count in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(count), ha="center", fontweight="bold")

# Plot 3: Calibration curve (coverage vs alpha)
ax3 = axes[2]
alphas_test = np.linspace(0.01, 0.5, 50)
coverages_test = []
for a_test in alphas_test:
    ql = min(math.ceil((n_cal + 1) * (1.0 - a_test)) / n_cal, 1.0)
    try:
        thr = float(np.quantile(scores, ql, method="higher"))
    except TypeError:
        thr = float(np.quantile(scores, ql, interpolation="higher"))
    cov = 0
    for i in range(n_cal):
        ps = build_prediction_set(cal_softmax[i], thr)
        if CLASS_LABELS[cal_labels_np[i]] in ps:
            cov += 1
    coverages_test.append(cov / n_cal)

ax3.plot(alphas_test, coverages_test, "b-", linewidth=2, label="Empirical Coverage")
ax3.plot(alphas_test, 1 - alphas_test, "r--", alpha=0.7, label="1 - alpha (target)")
ax3.axvline(alpha, color="green", linestyle=":", label=f"alpha={alpha}")
ax3.set_xlabel("Alpha (miscoverage rate)")
ax3.set_ylabel("Coverage")
ax3.set_title("Calibration Curve")
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0.4, 1.05)

plt.suptitle("Conformal Prediction Diagnostics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/conformal_diagnostics.png", dpi=150)
plt.show()

## Cell 13: Export Artifacts

In [ ]:
# ============================================================================
# Cell 13: Export Artifacts
# ============================================================================
import json

OUTPUT_DIR = Path("/kaggle/working")

# --------------------------------------------------------------------------
# 1. Model weights (CPU state dict)
# --------------------------------------------------------------------------
model_path = OUTPUT_DIR / "mini_gat.pt"
torch.save(model.cpu().state_dict(), str(model_path))
# Note: model is now on CPU after export
model_size = model_path.stat().st_size
print(f"1. Model saved:     {model_path}  ({model_size / 1024:.0f} KB)")

# --------------------------------------------------------------------------
# 2. Conformal calibration data
# --------------------------------------------------------------------------
cal_export = {
    "alpha": CONFIG["alpha"],
    "threshold": float(threshold),
    "n_calibration": int(n_cal),
    "empirical_coverage": float(empirical_coverage),
    "class_names": CLASS_LABELS,
    "singleton_rate": float(singleton_rate),
    "ambiguous_rate": float(ambiguous_rate),
    "mean_set_size": float(np.mean(set_sizes)),
    "score_stats": {
        "mean": float(scores.mean()),
        "std": float(scores.std()),
        "median": float(np.median(scores)),
        "min": float(scores.min()),
        "max": float(scores.max()),
    },
    "test_metrics": {
        "accuracy": float(test_metrics["accuracy"]),
        "precision": float(test_metrics["precision"]),
        "recall": float(test_metrics["recall"]),
        "f1": float(test_metrics["f1"]),
        "auc_roc": float(auc_roc),
    },
    "training_history": {
        "total_epochs": len(history["train_loss"]),
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "final_train_loss": float(history["train_loss"][-1]),
        "final_val_f1": float(history["val_f1"][-1]),
    },
}

cal_path = OUTPUT_DIR / "conformal_calibration.json"
with open(str(cal_path), "w") as f:
    json.dump(cal_export, f, indent=2)
print(f"2. Calibration saved: {cal_path}")

# --------------------------------------------------------------------------
# 3. Graph config (for inference consistency)
# --------------------------------------------------------------------------
graph_config = {
    "model_config": {
        k: v for k, v in CONFIG.items()
        if not isinstance(v, list) or k == "target_cwes"
    },
    "node_features": [
        "in_degree_norm",
        "out_degree_norm",
        "is_sink",
        "is_source",
        "depth_norm",
    ],
    "sink_patterns": SINK_PATTERNS,
    "source_patterns": SOURCE_PATTERNS,
    "max_nodes": CONFIG["max_nodes"],
    "embedding_model": CONFIG["embedding_model"],
    "torch_version": torch.__version__,
    "torch_geometric_version": torch_geometric.__version__,
    "dataset_info": {
        "total_samples": len(pyg_dataset),
        "train_samples": len(train_data),
        "val_samples": len(val_data),
        "cal_samples": len(cal_data),
        "n_cwes": len(set(tc["cwe"] for tc in test_cases)),
        "synthetic": USE_SYNTHETIC,
    },
}

config_path = OUTPUT_DIR / "graph_config.json"
with open(str(config_path), "w") as f:
    json.dump(graph_config, f, indent=2)
print(f"3. Config saved:    {config_path}")

print(f"\nAll artifacts saved to {OUTPUT_DIR}")

## Cell 14: Summary Report

In [ ]:
# ============================================================================
# Cell 14: Summary Report
# ============================================================================

total_time_min = training_time / 60

print(f"")
print(f"{'#'*70}")
print(f"#{'':68}#")
print(f"#{'SEC-C Mini-GAT Training Report':^68}#")
print(f"#{'':68}#")
print(f"{'#'*70}")
print(f"")
print(f"  DATASET")
print(f"  -------")
print(f"  Source:              {'Synthetic Juliet-like' if USE_SYNTHETIC else 'NIST Juliet Test Suite'}")
print(f"  Total files:         {len(test_cases)}")
print(f"  Graphs built:        {len(graph_data)}")
print(f"  PyG dataset size:    {len(pyg_dataset)}")
print(f"  CWEs covered:        {len(set(tc['cwe'] for tc in test_cases))}")
print(f"  Split:               train={len(train_data)}, val={len(val_data)}, cal={len(cal_data)}")
print(f"")
print(f"  MODEL")
print(f"  -----")
print(f"  Architecture:        MiniGAT (2-layer GAT)")
print(f"  Input dim:           {CONFIG['input_dim']} (768 GCB + 5 structural)")
print(f"  Hidden dim:          {CONFIG['hidden_dim']}")
print(f"  Output dim:          {CONFIG['output_dim']}")
print(f"  Attention heads:     L1={CONFIG['num_heads_l1']}, L2={CONFIG['num_heads_l2']}")
print(f"  Parameters:          {pc['total']:,}")
print(f"  Dropout:             {CONFIG['dropout']}")
print(f"")
print(f"  TRAINING")
print(f"  --------")
print(f"  Device:              {device}")
print(f"  Epochs completed:    {len(history['train_loss'])}")
print(f"  Best epoch:          {best_epoch}")
print(f"  Best val loss:       {best_val_loss:.4f}")
print(f"  Training time:       {total_time_min:.1f} min")
print(f"")
print(f"  METRICS (on calibration set)")
print(f"  ---------")
print(f"  Accuracy:            {test_metrics['accuracy']:.4f}")
print(f"  Precision:           {test_metrics['precision']:.4f}")
print(f"  Recall:              {test_metrics['recall']:.4f}")
print(f"  F1 Score:            {test_metrics['f1']:.4f}")
print(f"  AUC-ROC:             {auc_roc:.4f}")
print(f"")
print(f"  CONFORMAL PREDICTION")
print(f"  --------------------")
print(f"  Alpha:               {alpha}")
print(f"  Threshold:           {threshold:.4f}")
print(f"  Coverage:            {empirical_coverage:.4f} (target >= {1-alpha:.2f})")
print(f"  Singleton rate:      {singleton_rate:.1%}")
print(f"  Ambiguous rate:      {ambiguous_rate:.1%} (escalated to LLM)")
print(f"")
print(f"  OUTPUT FILES")
print(f"  ------------")
print(f"  1. mini_gat.pt                ({model_size/1024:.0f} KB)  - Model weights")
print(f"  2. conformal_calibration.json  - Calibration data")
print(f"  3. graph_config.json           - Inference config")
print(f"  4. juliet_graphs.pt            - PyG dataset")
print(f"  5. training_curves.png         - Loss/metric plots")
print(f"  6. evaluation_results.png      - Confusion/ROC")
print(f"  7. conformal_diagnostics.png   - Calibration plots")
print(f"  8. eda_analysis.png            - Dataset EDA")
print(f"")
print(f"  NEXT STEPS")
print(f"  ----------")
print(f"  1. Download mini_gat.pt and conformal_calibration.json")
print(f"  2. Place them in D:/sec-c/data/models/")
print(f"  3. Run the SEC-C pipeline with the trained model")
print(f"")
print(f"{'#'*70}")